# Text DataLoaders examples

> Build language-model and classification loaders from Hub datasets, text folders or in-memory documents

These examples use small samples to show the input format, cached metadata and resulting batches. Language models use documents directly; classifiers pair each document with a label.

In [7]:
from fastai.text.all import *
from fastai.text.tokenizer import *

Temporary sample folders and token caches stay available for this notebook session.

In [8]:
demo_tmp = tempfile.TemporaryDirectory(prefix='fastai-text-dls-')
demo_root = Path(demo_tmp.name)

## SimpleWiki: one article per row

SimpleWiki already has one article per row in `md`, so no preparation callable is needed. Two slices of its training source provide separate training and validation documents.

In [9]:
simple_dls = TextDataLoadersNew.from_dataset('answerdotai/simplewiki', subset='articles',
    text_col='md', train='train[:80]', valid='train[80:100]',
    cache_dir=demo_root/'simplewiki', bs=4, seq_len=32)
test_eq((len(simple_dls.train_ds), len(simple_dls.valid_ds)), (80, 20))
pd.read_csv(demo_root/'simplewiki/metadata.csv')['split'].value_counts()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

split
train[:80]       80
train[80:100]    20
Name: count, dtype: int64

A language-model batch pairs each token sequence with the next tokens in the stream.

In [10]:
x, y = simple_dls.one_batch()
test_eq(x[:, 1:], y[:, :-1])
simple_dls.show_batch(max_n=2, trunc_at=30)

<bos>**British English** is the dialect of English that is spoken in the United Kingdom. It is different in some ways from other types of English such as American
 April 27 - Independence Day (Sierra Leone and Togo)
- April 27 - Freedom Day (South Africa)
- April 27 - World Tapir
**British English** is the dialect of English that is spoken in the United Kingdom. It is different in some ways from other types of English such as American English
 27 - Independence Day (Sierra Leone and Togo)
- April 27 - Freedom Day (South Africa)
- April 27 - World Tapir Day


## WikiText-103: assemble documents first

WikiText stores article text across several rows. The preparation callable below joins those rows into documents and returns a metadata row for each article. Text stays separate from the metadata DataFrame.

In [11]:
_wiki_heading = re.compile(r'^ = [^=].* = \n$')

def get_wiki103_data(ds):
    "Assemble WikiText-103 articles and their metadata from source lines"
    docs, cur = [], []
    for text in ds['text']:
        if _wiki_heading.match(text):
            if cur: docs.append(''.join(cur))
            cur = []
        cur.append(text)
    if cur: docs.append(''.join(cur))
    texts = [d for d in docs if d.strip()]
    return texts, pd.DataFrame(dict(split=[str(ds.split)]*len(texts)))

In [12]:
wiki_dls = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:3000]', valid='validation',
    cache_dir=demo_root/'wikitext', bs=4, seq_len=32)
pd.read_csv(demo_root/'wikitext/metadata.csv')['split'].value_counts()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

split
validation      60
train[:3000]    57
Name: count, dtype: int64

In [13]:
wiki_dls.show_batch(max_n=2, trunc_at=30)

<bos> = Saves ; Sv % = 
 Save Percentage ; SO = Shutouts 
 † Denotes player spent time with another team before joining Blue
 , it was well received on its release , selling 3 @,@ 697 copies in its first week and appearing on several Billboard charts . Around this time ,
 = Saves ; Sv % = 
 Save Percentage ; SO = Shutouts 
 † Denotes player spent time with another team before joining Blue Jackets
 it was well received on its release , selling 3 @,@ 697 copies in its first week and appearing on several Billboard charts . Around this time , Townsend


## IMDb: language modeling with unlabeled reviews

Language modeling needs text, not sentiment labels. Hugging Face's `+` split expression combines a small sample of training reviews with unsupervised reviews. A separate test-source sample supplies validation documents.

In [14]:
imdb_lm = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text',
    train='train[:100]+unsupervised[:100]', valid='test[:40]',
    cache_dir=demo_root/'imdb-lm', bs=4, seq_len=32)
test_eq((len(imdb_lm.train_ds), len(imdb_lm.valid_ds)), (200, 40))
pd.read_csv(demo_root/'imdb-lm/metadata.csv')['split'].value_counts()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

split
train[:100]+unsupervised[:100]    200
test[:40]                          40
Name: count, dtype: int64

In [15]:
imdb_lm.show_batch(max_n=2, trunc_at=30)

<bos>As a kid I did think the weapon the murderer wielded was cool, however I was a kid and so I was a bit dumb. Even as a dumb
 that this will be the last movie from this series, but I doubt it. If there are more movies made, I wish Michael Landon, Jr and others
As a kid I did think the weapon the murderer wielded was cool, however I was a kid and so I was a bit dumb. Even as a dumb kid
 this will be the last movie from this series, but I doubt it. If there are more movies made, I wish Michael Landon, Jr and others would


For a new random split of every source, use `valid_pct` instead of named source slices. That includes the unsupervised source and any test data.

## IMDb: sentiment classification

Setting `label_col` pairs each document with its sentiment label: `0` is negative and `1` is positive. Samples from both ends of each source include both classes. Unsupervised reviews are left out.

In [16]:
imdb_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:50]+train[-50:]', valid='test[:20]+test[-20:]',
    cache_dir=demo_root/'imdb-clas', bs=4, seq_len=32)
test_eq(imdb_clas.vocab[1], ['0', '1'])
pd.read_csv(demo_root/'imdb-clas/metadata.csv').groupby(['split', 'label']).size()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

split                   label
test[:20]+test[-20:]    0        20
                        1        20
train[:50]+train[-50:]  0        50
                        1        50
dtype: int64

In [17]:
imdb_clas.show_batch(max_n=2, trunc_at=30)

,text,category
0,<bos>I have this film out of the library right now and I haven't finished watching it. It is so bad I am in disbelief. Audrey Hepburn had totally lost her,0
1,"<bos>WARNING: This review contains SPOILERS. Do not read if you don't want some points revealed to you before you watch the film.<br /><br />With a cast like this, you wonder",0


## IMDb files: standard folder layout

In `split/label/file` layouts, the folder names supply labels and source splits. We copy twenty real reviews per class into a small sample with the original layout. No metadata callable is needed.

In [21]:
imdb_path = untar_data(URLs.IMDB)
folder_path = demo_root/'imdb-files'
for split_name in ('train', 'test'):
    for label in ('neg', 'pos'):
        output = folder_path/split_name/label
        output.mkdir(parents=True, exist_ok=True)
        for file in get_text_files(imdb_path/split_name/label).sorted()[:20]: _ = (output/file.name).write_bytes(file.read_bytes())

In [22]:
folder_clas = TextDataLoadersNew.from_folder(folder_path, valid='test',
    cache_dir=demo_root/'imdb-files-cache', bs=4, seq_len=32)
test_eq(folder_clas.vocab[1], ['neg', 'pos'])
pd.read_csv(demo_root/'imdb-files-cache/metadata.csv').groupby(['split', 'label']).size()

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

split  label
test   neg      20
       pos      20
train  neg      20
       pos      20
dtype: int64

In [23]:
folder_clas.show_batch(max_n=2, trunc_at=30)

,text,category
0,<bos>Airport '77 starts as a brand new luxury 747 plane is loaded up with valuable paintings & such belonging to rich businessman Philip Stevens (James Stewart) who is flying them,neg
1,"<bos>THE NIGHT LISTENER (2006) **1/2 Robin Williams, Toni Collette, Bobby Cannavale, Rory Culkin, Joe Morton, Sandra Oh, John Cullum, Lisa Emery, Becky Ann Baker. (Dir: Patrick Stettner) <br /><br />Hitchcockian",pos


## In-memory documents: bring your own data

Data from a query, an API or another Python object can use the same loaders. Extract the texts and labels, tokenize the documents, and pass them to `from_docs` with their training and validation indices. This path writes no token cache.

In [24]:
records = [{'body': 'The soup needs more garlic.', 'topic': 'cooking'},
    {'body': 'The team scored twice.', 'topic': 'sport'},
    {'body': 'Bake the bread until golden.', 'topic': 'cooking'},
    {'body': 'The goalkeeper saved the final shot.', 'topic': 'sport'}]
records

[{'body': 'The soup needs more garlic.', 'topic': 'cooking'},
 {'body': 'The team scored twice.', 'topic': 'sport'},
 {'body': 'Bake the bread until golden.', 'topic': 'cooking'},
 {'body': 'The goalkeeper saved the final shot.', 'topic': 'sport'}]

In [25]:
texts = [record['body'] for record in records]
labels = [record['topic'] for record in records]
doc_tok = hf_tokenizer()
flat, offsets = tokenize_texts(texts, doc_tok)
docs = split_docs(flat, offsets)
doc_splits = ([0, 1], [2, 3])

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

Passing `labels` builds a classifier. Omitting them uses the same documents and split indices for language modeling.

In [26]:
docs

[array([  464, 17141,  2476,   517, 17644,    13], dtype=uint16), array([ 464, 1074, 7781, 5403,   13], dtype=uint16), array([   33,   539,   262,  8509,  1566, 10861,    13], dtype=uint16), array([  464, 26318,  7448,   262,  2457,  2823,    13], dtype=uint16)]

In [27]:
memory_clas = TextDataLoadersNew.from_docs(docs, doc_tok, doc_splits, labels=labels, bs=2, seq_len=8)
test_eq(memory_clas.vocab[1], ['cooking', 'sport'])
memory_clas.show_batch(max_n=2, trunc_at=30)

,text,category
0,<bos>The soup needs more garlic.<eos>,cooking
1,<bos>The team scored twice.<eos>,sport


In [28]:
memory_lm = TextDataLoadersNew.from_docs(docs, doc_tok, doc_splits, bs=1, seq_len=8)
memory_lm.show_batch(max_n=1, trunc_at=30)

<bos>The soup needs more garlic.<eos>
The soup needs more garlic.<eos><bos>


## A small ULMFiT training run

First learn to predict text on WikiText, adapt that language model to IMDb reviews, then transfer its encoder to a sentiment classifier. Small slices keep this run short. Each stage uses one ordinary `fit(1)` call with the default learning rate.

### Forward: train a language model

Use the article preparation function from earlier. We start a new AWD-LSTM because fastai's stock pretrained weights use a different tokenizer. The model's padding ID must match our tokenizer.

In [ ]:
wiki_train = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    cache_dir=demo_root/'train-wiki', bs=4, seq_len=32)
lm_config = dict(awd_lstm_lm_config, pad_token=wiki_train.pad_id)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [ ]:
learn = language_model_learner(wiki_train, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()], path=demo_root/'ulmfit')
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.527308    8.724760    0.049913  6153.399902 00:00     


### Forward: adapt the language model to IMDb

Assign the IMDb loaders to the same learner to continue training its WikiText weights on reviews. Both loaders use the same tokenizer and vocabulary. Sentiment labels are not needed here.

In [ ]:
imdb_train_lm = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text',
    train='train[:8]+unsupervised[:8]', valid='test[:4]',
    cache_dir=demo_root/'train-imdb-lm', bs=4, seq_len=32)
learn.dls = imdb_train_lm
learn.fit(1)
learn.save_encoder('imdb-forward')

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         8.628859    8.130865    0.035397  3397.737793 00:01     


### Forward: train a sentiment classifier

The classifier adds a new prediction head to the adapted encoder. Load that encoder, unfreeze the model and train on labeled reviews.

In [ ]:
imdb_train_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:8]+train[-8:]', valid='test[:4]+test[-4:]',
    cache_dir=demo_root/'train-imdb-clas', bs=4, seq_len=32)
clas_config = dict(awd_lstm_clas_config, pad_token=imdb_train_clas.pad_id)

 |████████████████████████████████████████| 100.00% [1/1 00:00<00:00]

In [ ]:
learn = text_classifier_learner(imdb_train_clas, AWD_LSTM, config=clas_config, seq_len=32,
    pretrained=False, metrics=accuracy, path=demo_root/'ulmfit')
learn.load_encoder('imdb-forward')
learn.unfreeze()
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  time    
0         0.823283    0.693227    0.500000  00:01     


In [ ]:
learn.show_results(max_n=2, trunc_at=30)

,text,category,category_
0,<bos>STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme),0,0
1,"<bos>Five minutes in, i started to feel how naff this was looking, you've got a completely unheroic hero and his overweight fool of a friend. Seen it all before, yeah",1,0


### Backward: train the language model

Repeat the same sequence with `backwards=True` on each loader. The cached token IDs and vocabulary are reused; the loader reverses documents before the model sees them. This direction gets its own model and encoder checkpoint.

In [ ]:
wiki_backward = TextDataLoadersNew.from_dataset('Salesforce/wikitext', subset='wikitext-103-raw-v1',
    get_data=get_wiki103_data, train='train[:30]', valid='validation[:20]',
    cache_dir=demo_root/'train-wiki', backwards=True, bs=4, seq_len=32)
learn = language_model_learner(wiki_backward, AWD_LSTM, config=lm_config, pretrained=False,
    metrics=[accuracy, Perplexity()], path=demo_root/'ulmfit')
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         9.402500    8.676561    0.046441  5863.848145 00:00     


### Backward: adapt to IMDb

Continue training the backward learner on the reversed IMDb documents, then save its encoder.

In [ ]:
imdb_backward_lm = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text',
    train='train[:8]+unsupervised[:8]', valid='test[:4]',
    cache_dir=demo_root/'train-imdb-lm', backwards=True, bs=4, seq_len=32)
learn.dls = imdb_backward_lm
learn.fit(1)
learn.save_encoder('imdb-backward')

epoch     train_loss  valid_loss  accuracy  perplexity  time    
0         8.610837    8.222059    0.037861  3722.159424 00:01     


### Backward: train the classifier

Use the backward encoder with backward classifier loaders. Labels stay unchanged, and decoding restores the text's original order for display.

In [ ]:
imdb_backward_clas = TextDataLoadersNew.from_dataset('stanfordnlp/imdb', text_col='text', label_col='label',
    train='train[:8]+train[-8:]', valid='test[:4]+test[-4:]',
    cache_dir=demo_root/'train-imdb-clas', backwards=True, bs=4, seq_len=32)
learn = text_classifier_learner(imdb_backward_clas, AWD_LSTM, config=clas_config, seq_len=32,
    pretrained=False, metrics=accuracy, path=demo_root/'ulmfit')
learn.load_encoder('imdb-backward')
learn.unfreeze()
learn.fit(1)

epoch     train_loss  valid_loss  accuracy  time    
0         0.719853    0.692866    0.500000  00:01     


In [ ]:
learn.show_results(max_n=2, trunc_at=30)

,text,category,category_
0,<bos>STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morning <br /><br />Former New Orleans homicide cop Jack Robideaux (Jean Claude Van Damme),0,0
1,"<bos>Five minutes in, i started to feel how naff this was looking, you've got a completely unheroic hero and his overweight fool of a friend. Seen it all before, yeah",1,0
